# Kata 09 — Reglas Condicionales por Ruta

> Spec: [`specs/009-path-rules/spec.md`](../../specs/009-path-rules/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(budget_calls=6)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Reglas heurísticas se cargan **sólo cuando el agente edita archivos que matchean un glob**. No siempre. Las reglas universales (seguridad) sí se cargan siempre.

## 2. Por qué importa

Cargar 2000 líneas de heurística Python en cada sesión cuando estoy editando un README desperdicia tokens en cada llamada y, peor, dispersa la atención del modelo (Kata 11).

## 3. Ejemplo correcto — activación condicional

In [ ]:
import fnmatch

UNIVERSAL_RULES = "## Reglas universales\n- Nunca commitees secretos.\n"

CONDITIONAL_RULES = [
    {"glob": "src/**/*.py",   "text": "## Python\n- type hints obligatorios\n- pytest, no unittest"},
    {"glob": "infra/**/*.tf", "text": "## Terraform\n- siempre `terraform fmt` antes\n- nada de provider inline"},
    {"glob": "docs/**/*.md",  "text": "## Docs\n- español, voz activa, sin jergon"},
]

def assemble_system(target_paths: list[str]) -> str:
    parts = [UNIVERSAL_RULES]
    for r in CONDITIONAL_RULES:
        if any(fnmatch.fnmatch(p, r["glob"]) for p in target_paths):
            parts.append(r["text"])
    return "\n".join(parts)

# Caso A: usuario edita Python
sys_A = assemble_system(["src/auth/login.py"])
print("=== sistema cuando se edita Python ===")
print(sys_A)
print("\n=== sistema cuando se edita README ===")
sys_B = assemble_system(["README.md"])
print(sys_B)

In [ ]:
resp_A = client.messages.create(system=sys_A, messages=[{"role": "user", "content": "Escribe un esqueleto de función login."}])
print("respuesta caso A:", next((b.text for b in resp_A.content if b.type == "text"), "")[:300])
print("input_tokens A:", resp_A.usage.input_tokens)

resp_B = client.messages.create(system=sys_B, messages=[{"role": "user", "content": "Escribe un esqueleto para el README del proyecto."}])
print("\nrespuesta caso B:", next((b.text for b in resp_B.content if b.type == "text"), "")[:300])
print("input_tokens B:", resp_B.usage.input_tokens)

`input_tokens` del caso B es estrictamente menor: las reglas de Python no se cargaron porque el target no las activa.

## 4. Anti-patrón — todas las reglas siempre activas

In [ ]:
sys_universal = UNIVERSAL_RULES + "\n".join(r["text"] for r in CONDITIONAL_RULES)
resp_bad = client.messages.create(system=sys_universal, messages=[{"role": "user", "content": "Escribe un esqueleto para el README del proyecto."}])
print("input_tokens (todo cargado):", resp_bad.usage.input_tokens)
print("vs caso B condicional:", resp_B.usage.input_tokens)

Diferencia: cada token extra del system prompt se paga **en cada turno** de la sesión.

## 5. Argumento de certificación

- **Universal vs condicional**: las reglas que aplican siempre (seguridad) cargan siempre; las heurísticas de lenguaje sólo cuando hay match.
- **El glob es del target del turno**, no del repo. Si el agente cambia de archivo, las reglas pueden cambiar (y el caché se invalida — costo aceptable porque la sesión completa rara vez salta dominios).
- **Conexión con otros katas**: los `CONDITIONAL_RULES` complementan la cascada del Kata 8; el ahorro de tokens habilita prefijo estable del Kata 10.

## 6. Auto-evaluación

**1. ¿La regla "no commits sin tests" es condicional o universal?**

Universal — aplica a todo cambio sin importar el lenguaje.

**2. ¿Qué pasa si dos reglas condicionales se aplican al mismo archivo?**

Ambas se concatenan al system prompt. Si se contradicen, es un bug del autor de las reglas — `assemble_system` debería detectar y advertir, o el equipo debe especializar el glob para no superponerse.

**3. ¿Cómo verifico que la regla NO se cargó cuando no debía?**

Comparando el system prompt resuelto entre dos targets: el caso B no contiene la sección "Python". Una aserción defensiva: `assert "type hints" not in assemble_system(["README.md"])`.